# ASPIRE Quickstart Notebook

This walkthrough shows how to load a pretrained ASPIRE model, run a single-record prediction, and (soon) score an entire dataset.

In [1]:
%pip install .

Processing /Users/edddddy/Documents/ASPIRE/open-source/aspire
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for aspire: filename=aspire-0.1-py3-none-any.whl size=21026 sha256=0196df5bc7f991bfd4fda549d011e744cf132bff71253b7ecd33ffd4d0b61620
  Stored in directory: /private/var/folders/79/h88rl61936g_75bvljbb2d200000gn/T/pip-ephem-wheel-cache-3vva7ldy/wheels/81/4c/2a/ac791919f82c85ffd6e3c8c889f123a212e772ab4e6109e4e1
Successfully built aspire
  Attempting uninstall: aspire
    Found existing installation: aspire 0.1
    Uninstalling aspire-0.1:
      Successfully uninstalled aspire-0.1

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. Load the pretrained model

In [2]:
from aspire import AspireModel

model = AspireModel.from_pretrained(repo_id="checkpoints/best_model", device="cpu")
model

/opt/homebrew/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /opt/homebrew/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


AspireModel(
  (model): ASPIREEnhanced(
    (semantic_grounding): SemanticFeatureGrounding(
      (bert): BertModel(
        (embeddings): BertEmbeddings(
          (word_embeddings): Embedding(30522, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (token_type_embeddings): Embedding(2, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): BertEncoder(
          (layer): ModuleList(
            (0-11): 12 x BertLayer(
              (attention): BertAttention(
                (self): BertSdpaSelfAttention(
                  (query): Linear(in_features=768, out_features=768, bias=True)
                  (key): Linear(in_features=768, out_features=768, bias=True)
                  (value): Linear(in_features=768, out_features=768, bias=True)
                  (dropout): Dropout(p=0.1, inplace=False)
                )
                (output): BertSe

## 2. Predict for a single record

In [ ]:
from aspire.model import Feature, Example

# 1) Describe your schema once
feature_specs = [
    {"name": "age", "description": "Age in years", "dtype": "continuous", "value_range": (18, 90)},
    {"name": "education_level", "description": "Education level", "dtype": "categorical", "choices": ["high_school", "bachelor", "master", "phd"]},
    {"name": "wellbeing_score", "description": "Wellbeing score (0-100)", "dtype": "continuous", "value_range": (0, 100)},
]
features = [Feature(**spec) for spec in feature_specs]

# 2) Define values for a single record
record = {
    "age": 29,
    "education_level": "master",
    "wellbeing_score": None,  # target
}

# 3) Pick which columns should be predicted
target_names = ["wellbeing_score"]
target_indices = [i for i, feat in enumerate(features) if feat.name in target_names]

# 4) Convert to the Example structure the model expects
values = [record.get(feat.name) for feat in features]
example = Example(
    features=features,
    values=values,
    target_indices=target_indices,
    dataset_context="Employee wellbeing survey",
    support_examples=None,
)

predictions = model.predict(example)
predictions


[44.87294554710388]

## 3. Predict for an entire dataset (without metadata)

In [4]:
import pandas as pd

# 1) Load your table to a pandas dataframe
dataset_path = "datasets/synthetic_federated_analysis_of_student_dropout_prediction_10.csv"
df = pd.read_csv(dataset_path)

# 2) Decide which columns to predict
target_columns = ["socioeconomic_index", "dropout_risk_score"]

# 3) Run inference; each entry in the output list matches one row in the DataFrame
predictions_list = model.predict_df(df, target_columns)
predictions_list


[[5.631790568551537, 0.45629952731258316],
 [5.612496473484109, 0.4545019659976066],
 [5.604508137405713, 0.45375772150957155],
 [5.649258823552505, 0.45792698168762136],
 [5.527296548098907, 0.44656419595054797],
 [5.5848514349822365, 0.4519263773738158],
 [5.62940367019268, 0.45607714859320464],
 [5.700356281580704, 0.46268754773252446],
 [5.657545750555871, 0.45869904481622237],
 [5.621209109754159, 0.45531369092462554]]

## 4. Adding metadata to dataset

In [5]:
import json
from pathlib import Path
from aspire.data_loader import add_metadata
from aspire.model import Feature

# 1) Load existing template
metadata_path = Path("datasets/metadata_federated_analysis_of_student_dropout_prediction_10.json")
metadata = json.loads(metadata_path.read_text())

In [6]:
# 2.1) Option 1: Put feature descriptions into List[str]
feature_desc = [feature.get("description", "") for feature in metadata["features"]]
feature_desc

['Number of days since enrollment',
 "Student's cumulative grade point average (GPA) scaled between 0-4",
 'Total number of credits registered per semester',
 'Percentage of classes attended in the last term',
 'Indicates receipt of financial assistance',
 'Student living situation',
 'Previous college attendance history',
 'Current probation status',
 'Prior course withdrawal incidents',
 'Standardized score reflecting socioeconomic background',
 'Days between end of last term and next enrollment',
 'Baseline risk assessment from institution']

In [7]:
# 2.2) Option 2: Put feature descriptions into List[Feature]
feature_desc = [
    Feature(
        name=item["name"],
        description=item.get("description", ""),
        dtype=("continuous" if (item.get("dtype") or item.get("type")) == "continuous" else "categorical"),
        choices=item.get("choices") or item.get("categories"),
        value_range=tuple(item.get("value_range") or item.get("range") or []) or None,
    )
    for item in metadata["features"]
]
feature_desc

[Feature(name='enrollment_duration', description='Number of days since enrollment', dtype='continuous', choices=None, value_range=(0.0, 365.0)),
 Feature(name='cumulative_gpa', description="Student's cumulative grade point average (GPA) scaled between 0-4", dtype='continuous', choices=None, value_range=(0.0, 4.0)),
 Feature(name='course_load', description='Total number of credits registered per semester', dtype='continuous', choices=None, value_range=(3.0, 21.0)),
 Feature(name='attendance_rate', description='Percentage of classes attended in the last term', dtype='continuous', choices=None, value_range=(0.0, 100.0)),
 Feature(name='financial_aid_status', description='Indicates receipt of financial assistance', dtype='categorical', choices=['none', 'partial', 'full'], value_range=None),
 Feature(name='residency_type', description='Student living situation', dtype='categorical', choices=['on_campus', 'commuter', 'online_only'], value_range=None),
 Feature(name='prior_college_enrollment'

In [8]:
# 3) Attach metadata to the DataFrame with add_metadata
df_with_meta = add_metadata(
    df.copy(),
    feature_desc=feature_desc,
    target=["dropout_risk_score"], # Optionally save target as metadata
    dataset_desc=metadata["description"],
)
# 4) Run inference using the metadata-aware DataFrame
meta_predictions = model.predict_df(df_with_meta)
meta_predictions

[[0.45778965950012207],
 [0.45254606008529663],
 [0.4512624144554138],
 [0.45955151319503784],
 [0.44750797748565674],
 [0.4515256881713867],
 [0.45854097604751587],
 [0.46702712774276733],
 [0.46700069308280945],
 [0.451099693775177]]

In [9]:
# Alternative: attach attrs-styled dict to metadata directly
dataset_path = "datasets/1592_Diabetes-Data-Set.csv"
df = pd.read_csv(dataset_path)
metadata_path = Path("datasets/metadata_1592_Diabetes-Data-Set.json")
metadata = json.loads(metadata_path.read_text())

df_with_meta = add_metadata(df.copy(), metadata=metadata)
df_with_meta.attrs

{'dataset_desc': 'Dataset description here',
 'target': [],
 'feature_desc': [{'name': 'Pregnancies',
   'description': '',
   'dtype': 'continuous',
   'choices': None,
   'value_range': [0.0, 1.0]},
  {'name': 'Glucose',
   'description': '',
   'dtype': 'continuous',
   'choices': None,
   'value_range': [0.0, 1.0]},
  {'name': 'BloodPressure',
   'description': '',
   'dtype': 'continuous',
   'choices': None,
   'value_range': [0.0, 1.0]},
  {'name': 'SkinThickness',
   'description': '',
   'dtype': 'continuous',
   'choices': None,
   'value_range': [0.0, 1.0]},
  {'name': 'Insulin',
   'description': '',
   'dtype': 'continuous',
   'choices': None,
   'value_range': [0.0, 1.0]},
  {'name': 'BMI',
   'description': '',
   'dtype': 'continuous',
   'choices': None,
   'value_range': [0.0, 1.0]},
  {'name': 'DiabetesPedigreeFunction',
   'description': '',
   'dtype': 'continuous',
   'choices': None,
   'value_range': [0.0, 0.9999999999999998]},
  {'name': 'Age',
   'description

## 5. Assisted metadata adding

In [10]:
from aspire.data_loader import create_metadata_template

dataset_path = "datasets/1592_Diabetes-Data-Set.csv"
metadata_path = create_metadata_template(dataset_path, overwrite=True)

Please edit metadata file at  datasets/metadata_1592_Diabetes-Data-Set.json


In [12]:
metadata = json.loads(Path(metadata_path).read_text())

df = pd.read_csv(dataset_path)
df_with_meta = add_metadata(df.copy(), metadata=metadata)

df_with_meta.attrs

{'dataset_desc': 'Dataset description here',
 'target': [],
 'feature_desc': [{'name': 'Pregnancies',
   'description': '',
   'dtype': 'continuous',
   'choices': None,
   'value_range': [0.0, 1.0]},
  {'name': 'Glucose',
   'description': '',
   'dtype': 'continuous',
   'choices': None,
   'value_range': [0.0, 1.0]},
  {'name': 'BloodPressure',
   'description': '',
   'dtype': 'continuous',
   'choices': None,
   'value_range': [0.0, 1.0]},
  {'name': 'SkinThickness',
   'description': '',
   'dtype': 'continuous',
   'choices': None,
   'value_range': [0.0, 1.0]},
  {'name': 'Insulin',
   'description': '',
   'dtype': 'continuous',
   'choices': None,
   'value_range': [0.0, 1.0]},
  {'name': 'BMI',
   'description': '',
   'dtype': 'continuous',
   'choices': None,
   'value_range': [0.0, 1.0]},
  {'name': 'DiabetesPedigreeFunction',
   'description': '',
   'dtype': 'continuous',
   'choices': None,
   'value_range': [0.0, 0.9999999999999998]},
  {'name': 'Age',
   'description